In [11]:
#import docx
import PyPDF2
import os
def read_text_file(file_path: str):
     """Read content from a txt file"""
     with open(file_path, 'r', encoding='utf-8') as file:
         return file.read()

def read_pdf_file(file_path: str):
    """Read content from a PDF file"""
    text = ""
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            text += page.extract_text() + "\n"
    return text

def read_docx_file(file_path: str):
     """Read content from a docx file"""
     doc = docx.Document(file_path)
     return "\n".join([paragraph.text for paragraph in doc.paragraphs])

In [12]:
def read_document(file_path: str):
    """Read document content based on file extension"""
    _, file_extension = os.path.splitext(file_path)
    file_extension = file_extension.lower()
    
    if file_extension == '.pdf':
        return read_pdf_file(file_path)
    elif file_extension == '.txt':
         return read_text_file(file_path)
    elif file_extension == '.docx':
        return read_docx_file(file_path)
    else:
        raise ValueError(f"Unsupported file format: {file_extension}")

text = read_document(r"docs\EmployeeHandbook.pdf")
print(text)

Em ployee Handbook Sam ple
Employment Relationship
Equal Employment Opportunity: [Company Name] is dedicated to
providing equal opportunities to all employees and applicants,
regardless of race, color, religion, sex, national origin, age, disability,
or any other protected status, in compliance with applicable laws. We
are firmly committed to fostering a diverse and inclusive workplace
where all individuals are valued, respected, and free from
discrimination or harassment. Our company celebrates diversity,
recognizing it as a source of strength. We embrace various
backgrounds, perspectives, and experiences, believing that a diverse
workforce enhances creativity, innovation, and success. We are
dedicated to upholding these values throughout our organization. 
Anti-Harassment: At [Company Name], we are dedicated to
maintaining a workplace that is safe, respectful, and free from all
forms of harassment. We believe that a positive work environment is
essential for the well-being and succes

###chunking


In [7]:
def split_text(text: str, chunk_size: int = 500):
    """Split text into chunks"""
    sentences = text.replace('\n', ' ').split('. ')
    print(sentences)
    chunks = []
    current_chunk = []
    current_size = 0
    print(f"no of sentences:  {len(sentences)}")
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if not sentence.endswith('.'):
            sentence += '.'

        sentence_size = len(sentence)

        if current_size + sentence_size > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
            current_size = sentence_size
        else:
            current_chunk.append(sentence)
            current_size += sentence_size

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

chunks = split_text(text)
print(f"Total chunks: {len(chunks)} ")
print(chunks[2])


['Em ployee Handbook Sam ple Employment Relationship Equal Employment Opportunity: [Company Name] is dedicated to providing equal opportunities to all employees and applicants, regardless of race, color, religion, sex, national origin, age, disability, or any other protected status, in compliance with applicable laws', 'We are firmly committed to fostering a diverse and inclusive workplace where all individuals are valued, respected, and free from discrimination or harassment', 'Our company celebrates diversity, recognizing it as a source of strength', 'We embrace various backgrounds, perspectives, and experiences, believing that a diverse workforce enhances creativity, innovation, and success', 'We are dedicated to upholding these values throughout our organization', ' Anti-Harassment: At [Company Name], we are dedicated to maintaining a workplace that is safe, respectful, and free from all forms of harassment', 'We believe that a positive work environment is essential for the well-be

In [8]:
import chromadb
from chromadb.utils import embedding_functions

In [9]:
client = chromadb.PersistentClient(path="./chroma_db")

# Use sentence-transformer embeddings for embedding our data
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(name="documents_collection", embedding_function=sentence_transformer_ef)

c:\Users\sivam\Desktop\Genai course Notes\RAG\Design\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2553.16it/s]


###Insert the data into ChromaDB

In [13]:
def process_document(file_path: str):
    """Process a single document and prepare it for ChromaDB"""
    try:
        # Read the document
        content = read_document(file_path)
        # Split into chunks
        chunks = split_text(content)

        # Prepare metadata
        file_name = os.path.basename(file_path)
        metadatas = [{"source": file_name, "chunk": i} for i in range(len(chunks))]
        ids = [f"{file_name}_chunk_{i}" for i in range(len(chunks))]

        return ids, chunks, metadatas
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return [], [], []

In [14]:
def add_to_collection(collection, ids, texts, metadatas):
    """Add documents to collection in batches"""
    if not texts:
        return

    batch_size = 100
    for i in range(0, len(texts), batch_size):
        end_idx = min(i + batch_size, len(texts))
        collection.add(
            documents=texts[i:end_idx],
            metadatas=metadatas[i:end_idx],
            ids=ids[i:end_idx]
        )

In [15]:
def process_and_add_documents(collection, folder_path: str):
      files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, file))]

      for file_path in files:
        print(f"Processing {os.path.basename(file_path)}...")
        ids, texts, metadatas = process_document(file_path)
        add_to_collection(collection, ids, texts, metadatas)
        print(f"Added {len(texts)} chunks to collection")

In [16]:
process_and_add_documents(collection, "./docs")

Processing EmployeeHandbook.pdf...
['Em ployee Handbook Sam ple Employment Relationship Equal Employment Opportunity: [Company Name] is dedicated to providing equal opportunities to all employees and applicants, regardless of race, color, religion, sex, national origin, age, disability, or any other protected status, in compliance with applicable laws', 'We are firmly committed to fostering a diverse and inclusive workplace where all individuals are valued, respected, and free from discrimination or harassment', 'Our company celebrates diversity, recognizing it as a source of strength', 'We embrace various backgrounds, perspectives, and experiences, believing that a diverse workforce enhances creativity, innovation, and success', 'We are dedicated to upholding these values throughout our organization', ' Anti-Harassment: At [Company Name], we are dedicated to maintaining a workplace that is safe, respectful, and free from all forms of harassment', 'We believe that a positive work envir

Semantic Search on chroma DB

In [17]:
def semantic_search(collection, query: str, n_results: int = 4):
    """Perform semantic search on the collection"""
    return collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["embeddings", "documents", "metadatas", "distances"]
    )

In [26]:
query = "What is the Leave Request policy?"
results = semantic_search(collection, query)
results


{'ids': [['EmployeeHandbook.pdf_chunk_27',
   'EmployeeHandbook.pdf_chunk_33',
   'EmployeeHandbook.pdf_chunk_10',
   'EmployeeHandbook.pdf_chunk_30']],
 'embeddings': [array([[ 0.04251879,  0.07140677,  0.02910287, ...,  0.0849264 ,
           0.02198543, -0.00418514],
         [ 0.00675166,  0.08516547,  0.09866308, ...,  0.08357203,
           0.06326576, -0.08392068],
         [ 0.00415372,  0.09810105,  0.03024802, ...,  0.06417054,
          -0.07565715, -0.01188253],
         [ 0.01744922,  0.08913551,  0.02230685, ...,  0.02398349,
           0.00749537,  0.03743728]], shape=(4, 384))],
 'documents': [['Employee Holidays and Leave Policy: Our employee holidays and leave policy outlines the procedures for requesting and taking time off for personal reasons, including holidays, special occasions, or other personal matters. Employees must submit leave requests through the approved channels, typically to their supervisor or the HR department. The application should include the date

In [27]:
def get_context_with_sources(results):
    """Get a combined context and formatted sources from search results."""
    # Combine the document chunks into a single context
    context = "\n\n".join(results['documents'][0])

    # Format the sources with metadata information
    sources = [f"{meta['source']} (chunk {meta['chunk']})" for meta in results['metadatas'][0]]

    return context, sources

context, sources = get_context_with_sources(results)
print(context)

Employee Holidays and Leave Policy: Our employee holidays and leave policy outlines the procedures for requesting and taking time off for personal reasons, including holidays, special occasions, or other personal matters. Employees must submit leave requests through the approved channels, typically to their supervisor or the HR department. The application should include the dates requested and the reason for the leave.

This policy outlines the notice period required for resignations and the reasons for termination, as well as the procedures involved.

Proper communication is vital to ensure coverage and maintain operational efficiency. Leave Requests: Planned absences, such as vacation or personal time off, should be requested in advance through the approved company channels. Work Hours and Attendance At [Company Name], we are dedicated to maintaining a structured work environment that values punctuality and attendance.

Employee Sick Leave Policy: Our employee sick leave policy provi

In [28]:
def get_context_with_sources(results):
    """Get a combined context and formatted sources from search results."""
    # Combine the document chunks into a single context
    context = "\n\n".join(results['documents'][0])

    # Format the sources with metadata information
    sources = [f"{meta['source']} (chunk {meta['chunk']})" for meta in results['metadatas'][0]]

    return context, sources

context, sources = get_context_with_sources(results)
print(context)

Employee Holidays and Leave Policy: Our employee holidays and leave policy outlines the procedures for requesting and taking time off for personal reasons, including holidays, special occasions, or other personal matters. Employees must submit leave requests through the approved channels, typically to their supervisor or the HR department. The application should include the dates requested and the reason for the leave.

This policy outlines the notice period required for resignations and the reasons for termination, as well as the procedures involved.

Proper communication is vital to ensure coverage and maintain operational efficiency. Leave Requests: Planned absences, such as vacation or personal time off, should be requested in advance through the approved company channels. Work Hours and Attendance At [Company Name], we are dedicated to maintaining a structured work environment that values punctuality and attendance.

Employee Sick Leave Policy: Our employee sick leave policy provi

Combinidng ChromaDB and Gemini for RAG

In [29]:
def get_prompt(query: str, context: str):
    """Prompt for Response Generation"""
    prompt = f"""Based on the following context, please answer the question.
    If the answer cannot be derived from the context, say "I cannot answer this based on the provided context."

    Context:
    {context}

    Question: {query}

    Answer:"""

    return prompt

In [30]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()


In [31]:
def generate_response_openai_style(query: str, context: str):
    """Generate a response using OpenAI"""

    prompt = get_prompt(query, context)
    print(prompt)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=500
    )

    return response.choices[0].message.content

In [32]:
def rag_query(collection, query: str, n_chunks: int = 2):
    """Perform RAG query: retrieve relevant chunks and generate answer"""
    # Get relevant chunks
    results = semantic_search(collection, query, n_chunks)
    context, sources = get_context_with_sources(results)

    # Generate response
    response = generate_response_openai_style(query, context)
    #response = generate_response_gemini_new(query, context)

    return response, sources


In [34]:
query = "What is the Leave Request policy?"
response, sources = rag_query(collection, query)

# Print results
#print("\nQuery:", query)
print("\nAnswer:", response)
#print("\nSources used:")
# for source in sources:
#     print(f"- {source}")

Based on the following context, please answer the question.
    If the answer cannot be derived from the context, say "I cannot answer this based on the provided context."

    Context:
    Employee Holidays and Leave Policy: Our employee holidays and leave policy outlines the procedures for requesting and taking time off for personal reasons, including holidays, special occasions, or other personal matters. Employees must submit leave requests through the approved channels, typically to their supervisor or the HR department. The application should include the dates requested and the reason for the leave.

This policy outlines the notice period required for resignations and the reasons for termination, as well as the procedures involved.

    Question: What is the Leave Request policy?

    Answer:

Answer: The Leave Request policy requires employees to submit leave requests through the approved channels, typically to their supervisor or the HR department. The application should includ